# 第 3 章 练习题答案

> 精选 2 道核心练习，巩固注意力机制。

## 练习 3.1：因果掩码验证

**题目**：手动构造注意力分数矩阵，应用因果掩码，验证掩码后未来 token 的注意力权重为 0。

In [ ]:
import torch
import torch.nn.functional as F

# 假设 4 个 token 的注意力分数（随机）
torch.manual_seed(42)
scores = torch.randn(4, 4)
print("原始注意力分数:")
print(scores.round(decimals=2))

# 构造因果掩码：上三角（未来）为 True
mask = torch.triu(torch.ones(4, 4), diagonal=1).bool()
print("\n因果掩码（True=屏蔽）:")
print(mask.int())

# 应用掩码 + softmax
scores_masked = scores.masked_fill(mask, -torch.inf)
weights = F.softmax(scores_masked, dim=-1)
print("\n掩码后的注意力权重:")
print(weights.round(decimals=3))
print("\n💡 上三角（未来 token）权重全为 0，每个 token 只关注自己和左侧。")
print("   例如 token 2（第 3 行）只对 token 0/1/2 有非零权重，token 3 为 0。")

## 练习 3.2：多头维度的影响

**题目**：保持 `d_out` 不变，改变 `num_heads`，观察输出形状是否变化。为什么？

In [ ]:
from src.gpt import MultiHeadAttention

batch, seq, d_in, d_out = 2, 6, 8, 8
x = torch.randn(batch, seq, d_in)

print(f"输入: {tuple(x.shape)}, d_out={d_out}\n")
print(f"{'num_heads':<12} {'head_dim':<12} {'输出形状':<20}")
print("-" * 44)
for n_heads in [1, 2, 4, 8]:
    mha = MultiHeadAttention(d_in, d_out, context_length=seq,
                             num_heads=n_heads, dropout=0.0)
    out = mha(x)
    print(f"{n_heads:<12} {d_out//n_heads:<12} {str(tuple(out.shape)):<20}")
print("\n💡 输出形状始终是 [b, seq, d_out]，与 num_heads 无关。")
print("   多头只是把 d_out 切成 n_heads 份并行算，每份 head_dim=d_out/n_heads。")
print("   头数影响的是'关注子空间的数量'，不是输出维度。")